In [ ]:
"""
Notebook 05 v3 — Model Definition & Training (Main Model)

Purpose:
    Train the L-TAE-based multi-task regression model on the prepared
    data and produce the reference baseline against which the experiments
    in Notebook 06 are compared.

What changes from v2:
    * Uses MaskedMultiTaskLoss: the regression heads only receive
      gradient from pixels that experienced at least one fire, matching
      the research goal of quantifying post-fire dynamics rather than
      general ecological assessment.
    * Reports per-land-cover breakdown of test-set predictions in
      addition to the recurrence-category breakdown.

Inputs:
    Processed tensors from Notebook 04 v3.

Outputs:
    model_main.pt, training_history.json, results_main.json,
    predictions_test.npz
"""

# Notebook 05 — Main model training

In [ ]:
# --- Environment auto-detection ---
import sys, os
IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    THESIS_ROOT = "/content/drive/MyDrive/thesisv4"
else:
    THESIS_ROOT = os.path.expanduser("~/thesisv4")

sys.path.insert(0, THESIS_ROOT)
print(f"=== Environment: {'COLAB' if IS_COLAB else 'LOCAL'} ===")
print(f"Workspace: {THESIS_ROOT}")

In [ ]:
import sys, importlib
# sys.path already set by auto-detect cell above
import config, model_defs, metrics
for m in (config, model_defs, metrics):
    importlib.reload(m)

config.ensure_dirs()
config.set_seed()
config.print_summary()

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.optim as optim
from sklearn.metrics import precision_recall_curve

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

## 1. Load processed data

In [ ]:
X        = np.load(config.FILES["X_scaled"])
y        = np.load(config.FILES["y_burned"])
mask     = np.load(config.FILES["X_scaled"].replace(".npy", "_mask.npy"))
targets  = dict(np.load(config.FILES["targets_aux"]))

with open(config.FILES["splits"]) as f:
    splits = json.load(f)
with open(config.FILES["feature_names"]) as f:
    feature_names = json.load(f)

idx_train = np.array(splits["train"])
idx_val   = np.array(splits["val"])
idx_test  = np.array(splits["test"])

N, T, C = X.shape
print(f"X={X.shape}  features={C}  T={T}")
print(f"Train/Val/Test: {len(idx_train)}/{len(idx_val)}/{len(idx_test)}")

## 2. Build data loaders

In [ ]:
train_loader = metrics.make_loader(
    X, y, mask, targets, idx_train,
    batch_size=config.TRAINING["batch_size"], shuffle=True)
val_loader   = metrics.make_loader(
    X, y, mask, targets, idx_val,
    batch_size=config.TRAINING["batch_size"], shuffle=False)
test_loader  = metrics.make_loader(
    X, y, mask, targets, idx_test,
    batch_size=config.TRAINING["batch_size"], shuffle=False)

## 3. Instantiate model, loss, optimiser

In [ ]:
model = model_defs.FireRecoveryModel(
    n_features=C,
    d_model=config.MODEL["d_model"],
    n_head=config.MODEL["n_head"],
    d_k=config.MODEL["d_k"],
    max_len=T,
    tau=config.MODEL["positional_tau"],
    dropout=config.MODEL["dropout"],
).to(DEVICE)

# MaskedMultiTaskLoss: regression heads only receive gradient from
# pixels that experienced at least one fire in the fire-reference inventory.
criterion = model_defs.MaskedMultiTaskLoss(
    alpha=config.LOSS["focal_alpha"],
    gamma=config.LOSS["focal_gamma"],
    lambda_aux=config.LOSS["lambda_aux"],
)
optimizer = optim.AdamW(
    model.parameters(),
    lr=config.TRAINING["lr"],
    weight_decay=config.TRAINING["weight_decay"],
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=config.TRAINING["epochs"])

n_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {n_params:,}")

## 4. Training loop

In [ ]:
history = {"train": [], "val": []}
best_val_auc = -1.0
best_state   = None

for epoch in range(1, config.TRAINING["epochs"] + 1):
    train_m = metrics.run_epoch(
        model, criterion, optimizer, train_loader, DEVICE,
        train=True, grad_clip=config.TRAINING["grad_clip"])
    val_m = metrics.run_epoch(
        model, criterion, optimizer, val_loader, DEVICE, train=False)
    scheduler.step()

    history["train"].append(train_m)
    history["val"].append(val_m)

    if val_m["auc"] > best_val_auc:
        best_val_auc = val_m["auc"]
        best_state   = {k: v.cpu().clone()
                        for k, v in model.state_dict().items()}

    if epoch == 1 or epoch % 5 == 0:
        print(f"epoch {epoch:3d}  "
              f"train_total={train_m['total']:.4f}  "
              f"val_total={val_m['total']:.4f}  "
              f"val_auc={val_m['auc']:.4f}")

print(f"\nBest val AUC: {best_val_auc:.4f}")

## 5. Threshold tuning + test-set evaluation

In [ ]:
model.load_state_dict(best_state)

best_thr, best_f1, val_probs, val_labels, val_mask_flat = metrics.tune_threshold(
    model, val_loader, DEVICE)
print(f"Best threshold: {best_thr:.3f}   F1: {best_f1:.3f}")

burned_metrics, correlations, predictions = metrics.evaluate_test_set(
    model, test_loader, DEVICE, best_thr, targets, idx_test)

## 6. Diagnostic plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot([h["total"] for h in history["train"]], label="train")
axes[0].plot([h["total"] for h in history["val"]],   label="val")
axes[0].set_title("Loss")
axes[0].set_xlabel("epoch")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot([h["auc"] for h in history["val"]])
axes[1].set_title(f"Val AUC (best = {best_val_auc:.3f})")
axes[1].set_xlabel("epoch")
axes[1].grid(True, alpha=0.3)

precision, recall, _ = precision_recall_curve(
    val_labels[val_mask_flat], val_probs[val_mask_flat])
axes[2].plot(recall, precision)
axes[2].set_xlabel("recall")
axes[2].set_ylabel("precision")
axes[2].set_title("Precision-Recall curve")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{config.PATHS['outputs_fig']}/training_curves.png",
            dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Per-recurrence-category distribution
meta_df = pd.read_csv(
    config.FILES["X_scaled"].replace(".npy", "_meta.csv"))

per_category = metrics.per_category_stats(
    predictions, idx_test, meta_df, config.PIXEL_TYPES)

colours = {"no_fires":          "#1D9E75",
           "low_recurrence":    "#EF9F27",
           "medium_recurrence": "#E24B4A",
           "high_recurrence":   "#8B0000"}

dimensions = {
    "severity":     predictions["severity"],
    "persistence":  predictions["persistence"],
    "recovery_gap": predictions["recovery_gap"],
}
pixel_types_test = meta_df.iloc[idx_test]["pixel_type"].values

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, (dim_name, values) in zip(axes, dimensions.items()):
    for ptype in config.PIXEL_TYPES:
        sel = pixel_types_test == ptype
        if sel.any():
            ax.hist(values[sel], bins=20, alpha=0.6,
                    label=ptype, color=colours[ptype])
    ax.set_xlabel(dim_name)
    ax.set_title(f"{dim_name} distribution")
    ax.grid(True, alpha=0.3)
axes[0].set_ylabel("count")
axes[-1].legend(loc="upper right", fontsize=8)

plt.tight_layout()
plt.savefig(f"{config.PATHS['outputs_fig']}/dimension_distributions.png",
            dpi=150, bbox_inches="tight")
plt.show()

## 7. Per-land-cover breakdown

For each dominant LC class in the test set, report the mean predicted
severity, persistence, and recovery_gap. This is the diagnostic view
that shows whether the model has absorbed the LC feature: if
post-agricultural (cropland/fallow) severity remains elevated, the
model is still confusing harvest cycles with fire signatures.

In [ ]:
lc_labels = meta_df["lc_dominant"].astype(int).values
per_lc = metrics.per_lc_stats(
    predictions, idx_test, lc_labels, config.LC_CATEGORIES)

metrics.print_lc_summary(per_lc, config.LC_CATEGORIES)

## 8. Persist model, history, results

In [ ]:
metrics.print_summary(
    "main_model_random_split", n_params,
    burned_metrics, correlations, per_category, config.PIXEL_TYPES)

import os
exp_dir = config.PATHS["models_main"]
torch.save(best_state, config.FILES["model_main"])
with open(config.FILES["history_main"], "w") as f:
    json.dump(history, f)

results_extra = {
    "training": {
        "n_epochs":         len(history["train"]),
        "final_train_loss": float(history["train"][-1]["total"]),
        "final_val_loss":   float(history["val"][-1]["total"]),
        "best_val_auc":     float(best_val_auc),
    },
    "model_info": {
        "n_parameters":   n_params,
        "d_model":        config.MODEL["d_model"],
        "n_head":         config.MODEL["n_head"],
        "d_k":            config.MODEL["d_k"],
        "positional_tau": config.MODEL["positional_tau"],
        "n_features":     C,
        "sequence_length": T,
        "loss_type":      "MaskedMultiTaskLoss",
    },
    "data_info": {
        "n_train": len(idx_train),
        "n_val":   len(idx_val),
        "n_test":  len(idx_test),
        "feature_names": feature_names,
        "split_type":    "random_stratified",
    },
}

with open(os.path.join(exp_dir, "results_main.json"), "w") as f:
    json.dump({
        "experiment_name":  "main_model_random_split",
        "burned_detection": burned_metrics,
        "correlations":     correlations,
        "per_category":     per_category,
        "per_lc":           per_lc,
        **results_extra,
    }, f, indent=2)

np.savez(
    config.FILES["preds_test"],
    severity=predictions["severity"],
    persistence=predictions["persistence"],
    recovery_gap=predictions["recovery_gap"],
    burned_probs=predictions["burned_probs"],
    burned_labels=predictions["burned_labels"],
    mask=predictions["mask"],
    indices=idx_test,
    threshold=best_thr,
)

print(f"\nSaved main-model results to {exp_dir}/results_main.json")